[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyTorch and TensorFlow, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)

# Recurrent Networks


## What you will be able to do

Turn a series of hourly readings into windows, forecast the next hour with an LSTM or a GRU in
PyTorch and in Keras, and give each library its sequences with the time axis where it expects it.

## The idea

### The problem

A weather station records a temperature every hour, and the next hour depends on the hours before
it: the rise and fall of the day, and the slow drift of the season. A dense layer given the last 24
readings sees 24 separate features, with nothing to say that the eleventh came after the tenth, and
it is fixed to that number of hours. A convolution would slide one detector along the readings, and
still look at only a few hours at a time.

What the forecast needs is a layer that reads the hours in order, one at a time, and keeps a summary
of what it has read so far.

### A formal definition

> A **recurrent layer** reads a sequence one step at a time and carries a **hidden state**, a vector
> that it updates at every step from that step's input and its own previous state. The state after
> the last step summarizes the whole sequence. An **LSTM** (long short-term memory) and a **GRU**
> (gated recurrent unit) are recurrent layers whose **gates** decide, at every step, how much of the
> old state to keep and how much of a new candidate state, computed from the step's input and the old
> state, to add.

### Why it works that way

The layer uses the same weights at every step, so it learns one rule for how a reading changes the
state, which works for the first hour of a window and the last alike. That is the same sharing as a
convolution's, along time instead of across a picture. The gates let it keep part of its state
unchanged from one step to the next, so something read early in a sequence can still count at the
end.

A sequence has three axes, the examples, the steps and the numbers at each step. Keras always takes
them as (batch, time, features). PyTorch's `nn.LSTM` takes (time, batch, features) unless it is made
with `batch_first=True`, and both orders are only numbers to it, so a batch given in the other order
is read without an error, as hours of the wrong sequences. The Common errors show what that does.

### Where this shows up

The readings are the **NumPy, Deep Dive** guide's for Oslo, made by the same formula. The
**Convolutional Networks** notebook asked the same question of which axis is which for pictures, and
text, sound and sensor logs are sequences that models read in the same (batch, time, features)
layout.

### What this notebook covers

- Oslo's readings cut into windows of 24 hours and the hour after, split by date
- a baseline that repeats the last reading
- what an LSTM returns, and a forecaster built on it in PyTorch
- a GRU in its place, and the same forecaster in Keras
- three errors, each about the axes of a sequence

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import torch

lstm = torch.nn.LSTM(input_size=1, hidden_size=4, batch_first=True)
sequences = torch.zeros(2, 5, 1)
output, (hidden, cell) = lstm(sequences)
print(output.shape, hidden.shape)
```

```
torch.Size([2, 5, 4]) torch.Size([1, 2, 4])
```

Two sequences of five steps, with one number at each step, go in. The LSTM's state has four numbers,
and it returns that state at every step, (2, 5, 4), and separately its state after the last step,
(1, 2, 4), whose first axis counts the layers.

## Setup

Three libraries, a way to quiet TensorFlow while it loads, and a year of readings.

- `math`, `datetime` and `timedelta` compute the readings, as in the **NumPy, Deep Dive** guide
- `numpy` cuts them into windows
- `torch` is PyTorch
- `tensorflow` is TensorFlow, and `keras` its library of layers and training tools
- `contextlib` and `os` hide the messages TensorFlow prints while it loads, as in the **Why PyTorch
  and TensorFlow** notebook

`year_of_readings` makes a made-up reading for every hour of 2025 at one station, from a formula with
a daily and a yearly cycle and a small fixed wobble, so the readings are the same on every run.
`oslo` holds Oslo's 8,760 readings, in degrees Celsius, as 32-bit floats. `NOVEMBER` is the position
of the first hour of 1 November, where the worked examples split the year.

In [1]:
import contextlib
import math
import os
from datetime import datetime, timedelta

import numpy as np
import torch


@contextlib.contextmanager
def c_messages_hidden():
    """Send what C and C++ code writes to the error stream to os.devnull while the block runs."""
    saved = os.dup(2)
    devnull = os.open(os.devnull, os.O_WRONLY)
    os.dup2(devnull, 2)
    try:
        yield
    finally:
        os.dup2(saved, 2)
        os.close(devnull)
        os.close(saved)


os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
with c_messages_hidden():
    import tensorflow as tf
    import keras


STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


oslo = np.array(year_of_readings("Oslo"), dtype="float32")
NOVEMBER = 24 * 304   # the hour 1 November begins, counted from midnight on 1 January

print("torch", torch.__version__.split("+")[0], "| tensorflow", tf.__version__, "| keras", keras.__version__,
      "|", len(oslo), "hourly readings at Oslo")

torch 2.11.0 | tensorflow 2.21.0 | keras 3.13.2 | 8760 hourly readings at Oslo


## Worked examples

### Oslo's readings

In [2]:
print(oslo.shape, oslo.dtype, "| coldest", oslo.min(), "| warmest", oslo.max())
print(oslo[:24])

(8760,) float32 | coldest -6.3 | warmest 19.3
[-3.5 -5.3 -5.3 -5.1 -4.7 -4.1 -5.1 -4.1 -3.1 -2.  -1.   0.  -0.7  0.
  0.6  1.   1.2  1.2 -0.6 -1.  -1.4 -1.9 -2.3 -4.5]


The first 24 readings are midnight to 11 p.m. on 1 January. They fall from −3.5 to the day's low in
the early morning and rise again into the afternoon, with the formula's wobble on top.

### Windows of 24 hours and the hour after

`sliding_window_view` lays the readings out in overlapping rows of 25, each starting an hour after
the one before. The first 24 numbers of a row are what the model reads, and the 25th is the hour it
forecasts.

In [3]:
windows = np.lib.stride_tricks.sliding_window_view(oslo, 25)
train_windows = windows[:NOVEMBER - 24]
test_windows = windows[NOVEMBER:]
print(windows.shape, "| training", len(train_windows), "| test", len(test_windows))

(8736, 25) | training 7272 | test 1440


The windows that end before 1 November are for training, and those that start on or after it are for
testing, so no test window shares an hour with a training window. Two neighboring windows share 23 of
their 24 hours, so a random split would test the model on near copies of what it trained on.

The readings are standardized with the training hours' mean and standard deviation, and the windows
become tensors of shape (windows, 24, 1), batch first, with one number at each step.

In [4]:
mean, std = float(oslo[:NOVEMBER].mean()), float(oslo[:NOVEMBER].std())


def as_tensors(rows):
    scaled_rows = (rows - mean) / std
    return (torch.from_numpy(np.ascontiguousarray(scaled_rows[:, :24])).unsqueeze(-1),
            torch.from_numpy(np.ascontiguousarray(scaled_rows[:, 24])).unsqueeze(-1))


X_train, y_train = as_tensors(train_windows)
X_test, y_test = as_tensors(test_windows)
print(round(mean, 2), round(std, 2), tuple(X_train.shape), tuple(y_train.shape))

7.69 6.65 (7272, 24, 1) (7272, 1)


`as_tensors` subtracts the mean, which makes a new array, and `np.ascontiguousarray` copies each slice
of it into an array of its own, so the tensors share no memory with `oslo`.

### A baseline that repeats the last reading

A forecast is only worth having if it beats the simplest one, that the next hour will be like the
last. Its error is measured in squared degrees, as the mean squared error of the test windows.

In [5]:
baseline_mse = ((test_windows[:, 23] - test_windows[:, 24]) ** 2).mean()
print("repeat the last reading:", round(float(baseline_mse), 3))

repeat the last reading: 0.729


Repeating the last reading has a mean squared error of 0.729 squared degrees over the test windows,
whose forecast hours run from 2 November to the end of December. Its square root, about 0.85
degrees, is the root mean square error, which weighs a large miss more heavily than an average of the
misses does.

### What an LSTM returns

In [6]:
torch.manual_seed(0)
lstm = torch.nn.LSTM(input_size=1, hidden_size=16, batch_first=True)
output, (hidden, cell) = lstm(X_train[:32])
print("output", tuple(output.shape), "| hidden", tuple(hidden.shape), "| cell", tuple(cell.shape))
print(torch.equal(output[:, -1], hidden[0]))

output (32, 24, 16) | hidden (1, 32, 16) | cell (1, 32, 16)
True


`input_size=1` is the one number at each step, and `hidden_size=16` the size of the state. `output`
holds the state after each of the 24 hours for each of the 32 windows, and `hidden` the state after
the last hour, with a first axis for the layer. The last step of `output` and `hidden[0]` are the
same numbers. An LSTM also carries a second vector, `cell`, which its gates use as longer-term
memory, and an output gate decides how much of it reaches `hidden`. A forecaster needs only the
state after the last hour.

### A forecaster in PyTorch

`Forecaster` runs the recurrent layer over the window and gives the last step's state to a `Linear`
layer, which turns its 16 numbers into one reading. `cell` chooses the kind of recurrent layer, so
the same class serves for a GRU. `train` takes the windows in shuffled batches of 64 for 8 epochs and
prints the test error every 2 epochs, in squared degrees, so it can be read beside the baseline's.

In [7]:
class Forecaster(torch.nn.Module):
    def __init__(self, cell=torch.nn.LSTM, batch_first=True):
        super().__init__()
        self.rnn = cell(input_size=1, hidden_size=16, batch_first=batch_first)
        self.head = torch.nn.Linear(16, 1)

    def forward(self, x):
        output, _ = self.rnn(x)
        return self.head(output[:, -1])


def train(model, epochs=8):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
    loss_fn = torch.nn.MSELoss()
    generator = torch.Generator().manual_seed(0)
    for epoch in range(1, epochs + 1):
        order = torch.randperm(len(X_train), generator=generator)
        for start in range(0, len(X_train), 64):
            rows = order[start:start + 64]
            optimizer.zero_grad()
            loss_fn(model(X_train[rows]), y_train[rows]).backward()
            optimizer.step()
        if epoch % 2 == 0:
            with torch.no_grad():
                test_mse = loss_fn(model(X_test), y_test).item() * std ** 2
            print(epoch, "test error", round(test_mse, 3))


torch.manual_seed(0)
lstm_forecaster = Forecaster()
print(sum(p.numel() for p in lstm_forecaster.parameters()), "parameters")
train(lstm_forecaster)

1233 parameters
2 test error 0.591
4 test error 0.414
6 test error 0.427
8 test error 0.419


The loss is computed on standardized readings, so multiplying it by the standard deviation squared
turns it back into squared degrees. After 8 epochs the LSTM's test error is 0.419, against the
baseline's 0.729. It rises a little between the fourth and the sixth epoch, from 0.414 to 0.427,
which training on shuffled batches can do from one epoch to the next.

### A GRU in place of the LSTM

A GRU has one state where an LSTM has two, and three sets of weights where an LSTM has four, so the
same forecaster has fewer parameters.

In [8]:
torch.manual_seed(0)
gru_forecaster = Forecaster(cell=torch.nn.GRU)
print(sum(p.numel() for p in gru_forecaster.parameters()), "parameters")
train(gru_forecaster)

929 parameters
2 test error 0.681
4 test error 0.435
6 test error 0.344
8 test error 0.304


The GRU forecaster has 929 parameters against the LSTM's 1,233, and its test error after 8 epochs is
0.304. Both beat the baseline, but this is one run of each from one seed, so it says little about
which layer is better for these readings.

### The same forecaster in Keras

Keras's `LSTM` takes (batch, time, features), with no setting for the order, and returns only the
state after the last step unless it is asked for every step with `return_sequences=True`. A `Dense`
layer then gives the forecast.

In [9]:
keras.utils.set_random_seed(0)
print(keras.layers.LSTM(16)(X_train[:32].numpy()).shape)
print(keras.layers.LSTM(16, return_sequences=True)(X_train[:32].numpy()).shape)

(32, 16)
(32, 24, 16)


In [10]:
keras.utils.set_random_seed(0)
lstm_keras = keras.Sequential([
    keras.Input((24, 1)),
    keras.layers.LSTM(16),
    keras.layers.Dense(1),
])
lstm_keras.compile(optimizer=keras.optimizers.Adam(0.005), loss="mse")
lstm_keras.fit(X_train.numpy(), y_train.numpy(), epochs=8, batch_size=64, verbose=0)

print(lstm_keras.count_params(), "parameters")
print("test error", round(lstm_keras.evaluate(X_test.numpy(), y_test.numpy(), verbose=0) * std ** 2, 3))

1169 parameters
test error 0.341


Keras's forecaster ends at 0.341, also below the baseline. It has 1,169 parameters, 64 fewer than
PyTorch's 1,233, and the weights of the two LSTM layers show where the difference comes from.

In [11]:
print("PyTorch:", [(name, tuple(p.shape)) for name, p in lstm_forecaster.rnn.named_parameters()])
print("Keras:  ", [tuple(w.shape) for w in lstm_keras.layers[0].weights])

PyTorch: [('weight_ih_l0', (64, 1)), ('weight_hh_l0', (64, 16)), ('bias_ih_l0', (64,)), ('bias_hh_l0', (64,))]
Keras:   [(1, 64), (16, 64), (64,)]


Both stack the weights of an LSTM's three gates and its candidate state into blocks of 4 × 16 = 64.
PyTorch keeps two bias vectors of 64, one added to the input's product and one to the state's, and
Keras keeps one, which accounts for the 64 parameters between the two counts. Keras also stores its
weight matrices turned around, (inputs, gates) where PyTorch has (gates, inputs).

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/13-recurrent-networks-solutions.ipynb).

**1.** Make an `nn.LSTM` with `hidden_size=8`, `num_layers=2` and `batch_first=True`, give it the first 5 training windows, and print the shapes of its output and of its hidden state.

In [12]:
# your code here

**2.** Print the square root of the baseline's test error and of `lstm_forecaster`'s, to read both in degrees.

In [13]:
# your code here

**3.** Cut the readings into windows of 6 hours and the hour after, train a `Forecaster` on them from `torch.manual_seed(0)`, and compare its test error with the 24-hour windows'.

In [14]:
# your code here

**4.** Build the Keras forecaster with a `GRU` layer in place of the `LSTM`, train it from `keras.utils.set_random_seed(0)`, and print its parameter count and test error.

In [15]:
# your code here

**5.** Forecast the first hour of 1 December, `oslo[24 * 334]`, from the 24 hours before it with `lstm_forecaster`, in degrees, and print it beside the reading and the baseline's forecast.

In [16]:
# your code here

## Common errors

### No error, and a model that reads the batch as time

The same `Forecaster`, made without `batch_first=True`, given the same batch of 32 windows. Then the
first window alone is changed, by adding 1 to every one of its hours, and the batch is given again.
A forecast made from each window alone would change for that window and no other.

In [17]:
batch = X_train[:32]
changed = batch.clone()
changed[0] += 1.0

torch.manual_seed(0)
batch_first = Forecaster()
torch.manual_seed(0)
time_first = Forecaster(batch_first=False)
with torch.no_grad():
    for name, model in [("batch_first=True ", batch_first), ("batch_first=False", time_first)]:
        moved = ((model(batch) - model(changed)).abs() > 1e-6).sum().item()
        print(name, tuple(model.rnn(batch)[0].shape), "| forecasts that moved by more than a millionth:",
              moved, "of 32")

batch_first=True  (32, 24, 16) | forecasts that moved by more than a millionth: 1 of 32
batch_first=False (32, 24, 16) | forecasts that moved by more than a millionth: 17 of 32


Both give an output of shape (32, 24, 16), so nothing looks wrong. Made without `batch_first=True`,
the LSTM read the batch as 32 steps of 24 sequences. The first window became the first step of every
sequence, so changing it moved 17 of the 32 forecasts by more than a millionth, and `output[:, -1]`
took the last of the 24 sequences at each step, where it should have taken the last hour of each
window. Make every recurrent layer with `batch_first=True` when the data is (batch, time, features).

### `RuntimeError: input.size(-1) must be equal to input_size. Expected 1, got 24`

Windows given without their feature axis, as 32 rows of 24 numbers.

In [18]:
lstm_forecaster(X_train[:32, :, 0])

RuntimeError: input.size(-1) must be equal to input_size. Expected 1, got 24

A two-dimensional input is read as one sequence, with no batch axis, so (32, 24) looked like 32
steps of 24 numbers, and the layer expected 1 number at each step. `unsqueeze(-1)` gives the windows
their feature axis back.

In [19]:
with torch.no_grad():
    print(tuple(lstm_forecaster(X_train[:32, :, 0].unsqueeze(-1)).shape))

(32, 1)


### `ValueError: Input 0 with name 'None' of layer 'lstm' is incompatible with the layer: expected ndim=3, found ndim=2. Full shape received: (None, 24)`

A Keras forecaster whose `Input` says each window is 24 numbers.

In [20]:
keras.utils.set_random_seed(0)
keras.Sequential([keras.Input((24,)), keras.layers.LSTM(16, name="lstm"), keras.layers.Dense(1)])

ValueError: Input 0 with name 'None' of layer 'lstm' is incompatible with the layer: expected ndim=3, found ndim=2. Full shape received: (None, 24)

`ndim=3` counts the batch, the steps and the features, and `(None, 24)` has no feature axis. Keras
raises this as it builds the model, before any data arrives. `keras.Input((24, 1))` gives each step
its one number, and the data needs the same axis, `[..., None]` on an array of windows.

In [21]:
keras.utils.set_random_seed(0)
print(keras.Sequential([keras.Input((24, 1)), keras.layers.LSTM(16), keras.layers.Dense(1)]).output_shape)

(None, 1)


## Recap

- You can cut a series into windows and the value after each, and split them by time.
- You can measure a forecast against the baseline of repeating the last value.
- You can build an LSTM or a GRU forecaster in PyTorch and in Keras, and use the state after the last
  step.
- You can give each library its sequences as (batch, time, features), and spot the model that read
  them in another order.

## What is next

The **Saving and Loading Models** notebook writes a trained model to a file in each library and
loads it back, so that training never has to be repeated to use it.

---

&#8592; **Previous:** [Convolutional Networks](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/12-convolutional-networks.ipynb)  &nbsp;·&nbsp;  [PyTorch and TensorFlow, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pytorch-and-tensorflow-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Saving and Loading Models](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pytorch-and-tensorflow-deep-dive/14-saving-and-loading-models.ipynb) &#8594;
